#### Visu du head

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

# 1. Configuration
path_source = "extraction_test_sas_sarl_2026_complete.parquet"
pd.set_option('display.max_columns', None)

# 2. Récupération des métadonnées (0% RAM)
parquet_file = pq.ParquetFile(path_source)
all_cols_raw = parquet_file.schema.names
total_rows = parquet_file.metadata.num_rows
total_cols = len(all_cols_raw)

print(f"📁 Analyse du fichier : {path_source}")
print(f"📊 Dimensions brutes : {total_rows} lignes x {total_cols} colonnes")
print("-" * 50)

# 3. Préparation des colonnes d'extrémités (20 premières / 20 dernières)
nb_inspect = 20
cols_debut = all_cols_raw[:nb_inspect]
cols_fin = all_cols_raw[-nb_inspect:]

# 4. Lecture flash (uniquement les 5 premières lignes)
df_debut = pd.read_parquet(path_source, columns=cols_debut).head(5)
df_fin = pd.read_parquet(path_source, columns=cols_fin).head(5)

# Nettoyage des noms pour l'affichage (enlève le caractère invisible BOM si présent)
df_debut.columns = [c.replace('\ufeff', '') for c in df_debut.columns]
df_fin.columns = [c.replace('\ufeff', '') for c in df_fin.columns]

# 5. AFFICHAGE DES CONTRÔLES
print(f"\n🔍 INSPECTION : LES {nb_inspect} PREMIÈRES COLONNES")
display(df_debut)

print(f"\n🔍 INSPECTION : LES {nb_inspect} DERNIÈRES COLONNES")
display(df_fin)

# 6. RÉSUMÉ TECHNIQUE DES COLONNES (Liste complète pour copier-coller si besoin)
print("\n📋 LISTE COMPLÈTE DES COLONNES DÉTECTÉES :")
print(df_debut.columns.tolist() + ["..."] + df_fin.columns.tolist())

---

### 1. Drop des colonnes inutiles

In [ ]:
# --- ÉTAPE DE NETTOYAGE (Suite directe du contrôle) ---

# AJOUT STRICT : On recrée la liste propre si le kernel a été redémarré
liste_propre = [c.replace('\ufeff', '') for c in all_cols_raw]

# 1. TA LISTE DE SUPPRESSION CONSOLIDÉE
colonnes_a_supprimer = [
    'libellecedexetablissement', 'libellecedex2etablissement', 'codecedexetablissement', 
    'codecedex2etablissement', 'distributionspecialeetablissement', 'distributionspeciale2etablissement',
    'l1_adressage_unitelegale', 'complementadresseetablissement', 'complementadresse2etablissement',
    'numerovoieetablissement', 'numerovoie2etablissement', 'typevoieetablissement', 
    'typevoie2etablissement', 'libellevoieetablissement', 'libellevoie2etablissement',
    'indicerepetitionetablissement', 'indicerepetition2etablissement',
    'codecommune2etablissement', 'libellecommune2etablissement', 'codepostal2etablissement',
    'adresseetablissement', 'libellecommuneetablissement', 'departementetablissement', 'regionetablissement',
    'codepaysetrangeretablissement', 'libellepaysetrangeretablissement', 
    'codepaysetranger2etablissement', 'libellepaysetranger2etablissement',
    'libellecommuneetrangeretablissement', 'libellecommuneetranger2etablissement',
    'sexeunitelegale', 'prenom1unitelegale', 'prenom2unitelegale', 'prenom3unitelegale', 
    'prenom4unitelegale', 'prenomusuelunitelegale', 'pseudonymeunitelegale', 
    'nomunitelegale', 'nomusageunitelegale', 
    'nic', 'nicsiegeunitelegale', 'siretsiegeunitelegale', 'siret', 'etablissementsiege', 
    'statutdiffusionetablissement', 'statutdiffusionunitelegale', 'unitepurgeeunitelegale', 
    'identifiantassociationunitelegale', 'nombreperiodesetablissement', 'nombreperiodesunitelegale', 
    'sectionetablissement', 'soussectionetablissement', 'divisionetablissement', 
    'groupeetablissement', 'classeetablissement', 'sectionunitelegale', 
    'soussectionunitelegale', 'divisionunitelegale', 'groupeunitelegale', 'classeunitelegale',
    'nomenclatureactiviteprincipaleunitelegale', 'nomenclatureactiviteprincipaleetablissement',
    'activiteprincipaleregistremetiersetablissement',
    'codeepciunitelegale', 'codeepcietablissement', 'libelleepciunitelegale', 'libelleepcietablissement',
    'epcietablissement', 'epciunitelegale',
    'anneeeffectifsetablissement', 'anneeeffectifsunitelegale', 'activiteprincipaleetablissement',
    'trancheeffectifsetablissement', 'trancheeffectifsetablissementtriable',
    'trancheeffectifsunitelegaletriable', 'categorieentreprise', 'anneecategorieentreprise',
    'societemissionunitelegale', 'caractereemployeurunitelegale', 'sigleunitelegale',
    'codearrondissementetablissement', 'denominationusuelleetablissement',
    'denominationusuelle1unitelegale', 'denominationusuelle2unitelegale', 'denominationusuelle3unitelegale',
    'enseigne1etablissement', 'enseigne2etablissement', 'enseigne3etablissement',
    'datederniertraitementetablissement', 'datederniertraitementunitelegale',
    'datedebutetablissement', 'datedbutunitelegale', 'naturejuridiqueunitelegale', 
    'naturejuridiqueetablissement', 'caractereemployeuretablissement'
]

# 2. On filtre la liste globale pour définir les colonnes à garder
colonnes_a_garder = [c for c in liste_propre if c not in colonnes_a_supprimer]

# 3. Préparation de la lecture
col_siren_brute = next(c for c in all_cols_raw if 'siren' in c)
colonnes_lecture = [col_siren_brute if c == 'siren' else c for c in colonnes_a_garder]

# 4. Chargement
print(f"⏳ Chargement de {len(colonnes_a_garder)} colonnes...")
df_clean = pd.read_parquet(path_source, columns=colonnes_lecture)

# 5. Remise au propre
df_clean.columns = colonnes_a_garder
df_clean = df_clean.reset_index(drop=True)

print(f"🚀 Succès ! Dataset nettoyé : {df_clean.shape}")
display(df_clean.head())

---

### 1.1 Inspection valeurs nulles

In [ ]:
# --- BILAN DE SANTÉ DU DATASET NETTOYÉ ---

print(f"Analyse des manquants sur {df_clean.shape[1]} colonnes et {len(df_clean)} lignes...")

# Calcul rapide des NaN et des chaînes vides
def get_missing_stats(df):
    results = []
    for col in df.columns:
        # 1. Compte des NaN/NaT
        nb_nan = df[col].isna().sum()
        
        # 2. Compte des chaînes vides (uniquement si la colonne est de type objet/string)
        nb_empty = 0
        if df[col].dtype == 'object':
            nb_empty = (df[col].astype(str).str.strip() == '').sum()
            
        total_missing = nb_nan + nb_empty
        
        results.append({
            'Colonne': col,
            'Nb_NaN': total_missing,
            '%_NaN': round((total_missing / len(df)) * 100, 2)
        })
    return pd.DataFrame(results)

# Génération du rapport
df_missing_report = get_missing_stats(df_clean)

# Affichage complet
pd.set_option('display.max_rows', None)
print("\n📊 TAUX DE VALEURS NULLES PAR COLONNE :")
display(df_missing_report.sort_values(by='%_NaN', ascending=False))

---

### 2. Labeling de la cible (Target) et consolidation des dates de fermeture

In [ ]:
# --- ÉTAPE DE FIABILISATION ANALYTIQUE ---

# 1. Conversion des colonnes en format Date
df_clean["datefermetureetablissement"] = pd.to_datetime(df_clean["datefermetureetablissement"], errors="coerce")
df_clean["datefermetureunitelegale"] = pd.to_datetime(df_clean["datefermetureunitelegale"], errors="coerce")
df_clean["datedebutunitelegale"] = pd.to_datetime(df_clean["datedebutunitelegale"], errors="coerce")

# 2. Création de la Date Finale Consolidée
df_clean["Date_fermeture_finale"] = df_clean[["datefermetureetablissement", "datefermetureunitelegale"]].max(axis=1)

# 3. Création de l'indicateur 'est_fermee'
df_clean['est_fermee'] = (
    (df_clean["Date_fermeture_finale"].notna()) | 
    (df_clean['etatadministratifetablissement'] == 'Fermé') |
    (df_clean['etatadministratifunitelegale'] != 'Active')
).astype(int)

# 4. Suppression des "Fantômes" (Type CHBN)
date_coupure = pd.to_datetime('2011-01-01')
mask_fantomes = (
    (df_clean['est_fermee'] == 0) & 
    (df_clean['datedebutunitelegale'] < date_coupure) & 
    (df_clean['trancheeffectifsunitelegale'] == 'Etablissement non employeur')
)

df_clean = df_clean[~mask_fantomes].copy()

# 5. BILAN & NETTOYAGE DES COLONNES SOURCES
# On supprime la colonne administrative car 'est_fermee' la remplace avantageusement
df_clean.drop(columns=['etatadministratifetablissement'], inplace=True)

print(f"🧹 Nettoyage des fantômes terminé : {mask_fantomes.sum()} lignes supprimées.")
print(f"🗑️ Colonne 'etatadministratifetablissement' supprimée.")
print(f"📊 Volume final : {len(df_clean)} lignes.")
print(f"📈 Taux de fermeture réconcilié : {(df_clean['est_fermee'].mean() * 100):.2f}%")

# 6. APERÇU DE CONTRÔLE (Mis à jour sans la colonne supprimée)
cols_check = ['siren', 'trancheeffectifsunitelegale', 'Date_fermeture_finale', 'est_fermee']
display(df_clean[cols_check].head(10))

---

### 3. Création des colonnes age estimé et age incertain

In [ ]:
# --- CALCUL DE L'ÂGE ESTIMÉ AVEC NETTOYAGE (VERSION OPTIMISÉE) ---

# 1. Configuration
cols_dates_naissance = ["datecreationunitelegale", "datedebutunitelegale", "datecreationetablissement"]
date_ref = pd.Timestamp("2026-05-11") 
borne_min = pd.Timestamp("1900-01-01")

# 2. Nettoyage et conversion des dates de naissance
for col in cols_dates_naissance:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")
    df_clean.loc[df_clean[col] < borne_min, col] = pd.NaT

# 3. FILTRE DE SÉCURITÉ : Suppression des "entreprises fantômes"
# On supprime AVANT le calcul pour éviter d'attribuer un âge de 36 ans aux boîtes de 1994
mask_fantomes = (df_clean['est_fermee'] == 1) & (df_clean['Date_fermeture_finale'].isna())
df_clean = df_clean.drop(df_clean[mask_fantomes].index)
print(f"🧹 Nettoyage : {mask_fantomes.sum():,} entreprises fantômes supprimées.")

# 4. Détermination de la date de "fin" réelle
# Le fillna(date_ref) ne s'applique désormais qu'aux boîtes ouvertes (le bon usage)
date_fin_effective = df_clean["Date_fermeture_finale"].fillna(date_ref)

# 5. Calcul des âges intermédiaires en ANNÉES
df_clean["age_ul_y"] = (date_fin_effective - df_clean["datecreationunitelegale"]).dt.days / 365.25
df_clean["age_debut_y"] = (date_fin_effective - df_clean["datedebutunitelegale"]).dt.days / 365.25
df_clean["age_etab_y"] = (date_fin_effective - df_clean["datecreationetablissement"]).dt.days / 365.25

# 6. Estimation finale de l'âge
df_clean["age_estime"] = df_clean[["age_ul_y", "age_debut_y", "age_etab_y"]].max(axis=1)
df_clean["age_estime"] = df_clean["age_estime"].fillna(0).clip(lower=0).round(1).astype('float32')

# 7. Indicateur de fiabilité et nettoyage des colonnes temporaires
df_clean["age_incertain"] = df_clean[cols_dates_naissance].isna().all(axis=1).astype('int8')
df_clean.drop(columns=["age_ul_y", "age_debut_y", "age_etab_y"], inplace=True)

# --- VÉRIFICATION FINALE ---
print(f"✅ Calcul terminé sur {len(df_clean):,} lignes.")
print(f"📊 Nouvelle moyenne d'âge : {df_clean['age_estime'].mean():.1f} ans.")

---

### 4. Drop des nouvelles colonnes inutiles

In [ ]:
# --- NETTOYAGE DES COLONNES SOURCES APRÈS CALCULS ---

colonnes_a_retirer = [
    'datecreationetablissement',
    'datecreationunitelegale',
    'datedebutunitelegale',
    'etatadministratifunitelegale',
    'datefermetureetablissement',
    'datefermetureunitelegale'
]

# On ne supprime que celles qui existent encore dans le df pour éviter les erreurs
df_clean = df_clean.drop(columns=[c for c in colonnes_a_retirer if c in df_clean.columns])

print(f"✅ Nettoyage terminé. Nouvelles dimensions : {df_clean.shape}")
display(df_clean.head())

---

### 5. Traitement des effectifs par tranches

In [ ]:
# --- REGROUPEMENT ET ENCODAGE DES EFFECTIFS ---

mapping_effectifs = {
    # Micro-entreprises / Indépendants
    'Etablissement non employeur': 0,
    '0 salarié': 0,
    
    # Très Petites Entreprises (TPE)
    '1 ou 2 salariés': 1,
    '3 à 5 salariés': 2,
    '6 à 9 salariés': 3,
    
    # Petites Entreprises (PE)
    '10 à 19 salariés': 4,
    '20 à 49 salariés': 5,
    
    # Moyennes Entreprises (ME)
    '50 à 99 salariés': 6,
    '100 à 199 salariés': 7,
    '200 à 249 salariés': 8,
    
    # Entreprises de Taille Intermédiaire (ETI)
    '250 à 499 salariés': 9,
    '500 à 999 salariés': 10,
    
    # Grandes Entreprises (GE)
    '1 000 à 1 999 salariés': 11,
    '2 000 à 4 999 salariés': 11,
    '5 000 à 9 999 salariés': 11,
    '10 000 salariés et plus': 11
}

# Application du mapping
df_clean['effectifs'] = df_clean['trancheeffectifsunitelegale'].map(mapping_effectifs)

# Gestion des valeurs manquantes (on les met souvent dans la catégorie la plus fréquente ou -1)
df_clean['effectifs'] = df_clean['effectifs'].fillna(0).astype(int)

# On peut maintenant supprimer la colonne textuelle originale
df_clean.drop(columns=['trancheeffectifsunitelegale'], inplace=True)

print("✅ Tranches d'effectifs encodées.")
print(df_clean['effectifs'].value_counts().sort_index())

---

### 6. Traitement Activités

In [ ]:
# 1. Chargement du référentiel APE
ref_ape = pd.read_csv("codes_ape.csv", dtype={'code_ape': str})

# 2. Fonction de formatage robuste (ex: 452V -> 45.20V)
def format_ape_full(code):
    if pd.isna(code): return code
    val = str(code).replace('.', '') # On enlève les points existants
    
    # Si le code ressemble à 452V (4 caractères), on insère le 0 manquant
    if len(val) == 4 and val[-1].isalpha():
        val = f"{val[:3]}0{val[3]}" # Devient 4520V
        
    # On place ensuite le point après les 2 premiers chiffres
    if len(val) >= 3:
        return f"{val[:2]}.{val[2:]}"
    return val

# 3. Application du formatage sur la colonne principale
df_clean['activiteprincipaleunitelegale'] = df_clean['activiteprincipaleunitelegale'].apply(format_ape_full)

# 4. Création de la colonne 'code_ape' (2 chiffres) pour la jointure
df_clean['code_ape'] = df_clean['activiteprincipaleunitelegale'].str[:2]

# 5. Nettoyage des colonnes existantes pour éviter les doublons au merge
cols_to_drop = ['libelle_section_ape', 'libelle_ape']
df_clean = df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns], errors='ignore')

# 6. Mapping avec le référentiel
df_clean = df_clean.merge(ref_ape[['code_ape', 'libelle_ape']], 
                         on='code_ape', 
                         how='left')

# 7. Renommage final
df_clean = df_clean.rename(columns={'libelle_ape': 'libelle_section_ape'})

# Affichage pour vérification
print(df_clean[['siren', 'activiteprincipaleunitelegale', 'code_ape', 'libelle_section_ape']].head())

---

### 7. Traitement ESS

In [ ]:
# Transformation de la colonne ESS en binaire (0 ou 1)

df_clean['economiesocialesolidaireunitelegale'] = df_clean['economiesocialesolidaireunitelegale'].apply(
    lambda x: 1 if str(x).strip().upper() == 'O' else 0
)

# On s'assure que le type est bien entier pour le modèle
df_clean['economiesocialesolidaireunitelegale'] = df_clean['economiesocialesolidaireunitelegale'].astype(int)

# Vérification rapide sur les premières lignes
print(df_clean[['denominationunitelegale', 'economiesocialesolidaireunitelegale']].head())

---

### 8. Traitement coordonnées GPS

In [ ]:
import numpy as np

# 1. On sépare la colonne en deux (expand=True crée un DataFrame de 2 colonnes)
split_geoloc = df_clean['geolocetablissement'].str.split(',', expand=True)

# 2. On remplace les chaînes vides ou espaces par NaN pour éviter l'erreur de conversion
split_geoloc = split_geoloc.replace(['', ' ', None], np.nan)

# 3. On convertit en float et on assigne aux nouvelles colonnes
df_clean[['latitude', 'longitude']] = split_geoloc.astype(float)

# 4. Suppression de la colonne d'origine
df_clean = df_clean.drop(columns=['geolocetablissement'], errors='ignore')

# Vérification
print(df_clean[['siren', 'latitude', 'longitude']].head())

---

### 9. Contrôle et traitement des valeurs nulles

In [ ]:
# Contrôle des valeurs nulles

print(f"🔍 Analyse exhaustive de df_clean : {df_clean.shape[0]} lignes | {df_clean.shape[1]} colonnes")

# Initialisation des compteurs
stats_finales = []

for col in df_clean.columns:
    # 1. Détection des NaN / NaT (tous types confondus)
    nb_nan = df_clean[col].isna().sum()
    
    # 2. Détection des chaînes vides (uniquement pour les colonnes de type 'object' ou 'string')
    nb_empty = 0
    if df_clean[col].dtype == 'object' or str(df_clean[col].dtype) == 'string':
        # On convertit en string et on strip pour attraper " ", "" ou les espaces insécables
        nb_empty = (df_clean[col].astype(str).str.strip().isin(['', 'nan', 'None', 'NaT'])).sum()
    
    total_invalide = nb_nan + nb_empty
    
    stats_finales.append({
        'Colonne': col,
        'Type': str(df_clean[col].dtype),
        'Valeurs_Nulles': total_invalide,
        '%_Nulles': round((total_invalide / len(df_clean)) * 100, 2)
    })

# Création du DataFrame de diagnostic
df_bilan_final = pd.DataFrame(stats_finales)

print("\n📊 BILAN DE SANTÉ FINAL DU DATASET :")

# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_rows', None)

# On affiche tout, trié par le taux de nullité décroissant
display(df_bilan_final.sort_values(by='%_Nulles', ascending=False))

# Petit récapitulatif rapide
nb_colonnes_propres = len(df_bilan_final[df_bilan_final['Valeurs_Nulles'] == 0])
print(f"\n✅ {nb_colonnes_propres} colonnes sont parfaitement remplies (0% nulles).")

In [ ]:
# --- NETTOYAGE DES LIGNES COMPORTANT DES VALEURS NULLES ---

# 1. On définit les colonnes à ignorer pour la suppression (on garde les entreprises actives)
colonnes_a_ignorer = ['Date_fermeture_finale']

# 2. On récupère la liste des colonnes sur lesquelles on veut tester la présence de nuls
colonnes_test = [c for c in df_clean.columns if c not in colonnes_a_ignorer]

# 3. Suppression des lignes (dropna)
# On ne regarde que les colonnes dans colonnes_test
taille_avant = len(df_clean)
df_clean = df_clean.dropna(subset=colonnes_test)

# 4. Nettoyage des résidus (chaînes vides éventuelles dans les colonnes object)
# Parfois dropna ne voit pas les "" ou " ", on s'assure de les traiter
for col in df_clean.select_dtypes(include=['object']).columns:
    if col != 'Date_fermeture_finale':
        df_clean = df_clean[df_clean[col].astype(str).str.strip() != '']

taille_apres = len(df_clean)

print(f"🧹 Nettoyage terminé.")
print(f"📉 Lignes supprimées : {taille_avant - taille_apres}")
print(f"✅ Lignes restantes : {taille_apres}")

# Vérification rapide
print("\n--- Vérification des nuls restants ---")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

---

### 10. Types des colonnes

In [ ]:
# --- OPTIMISATION FINALE DU TYPAGE ---

# 1. Identifiants et texte libre (on reste en object/str)
df_clean['siren'] = df_clean['siren'].astype(str)
df_clean['denominationunitelegale'] = df_clean['denominationunitelegale'].astype(str)

# 2. Catégories (Gain RAM massif + prêt pour XGBoost)
cols_cat = [
    'codepostaletablissement',
    'codecommuneetablissement',
    'categoriejuridiqueunitelegale',
    'activiteprincipaleunitelegale',
    'codedepartementetablissement',
    'coderegionetablissement',
    'code_ape',
    'libelle_section_ape'
]
for col in cols_cat:
    df_clean[col] = df_clean[col].astype('category')

# 3. Précision flottante optimisée (float64 -> float32)
for col in ['age_estime', 'latitude', 'longitude']:
    df_clean[col] = df_clean[col].astype('float32')

# 4. Entiers optimisés (Moins de bits pour les petites valeurs)
# ESS, est_fermee et age_incertain sont binaires (0/1) -> int8 suffit amplement
df_clean['economiesocialesolidaireunitelegale'] = df_clean['economiesocialesolidaireunitelegale'].astype('int8')
df_clean['est_fermee'] = df_clean['est_fermee'].astype('int8')
df_clean['age_incertain'] = df_clean['age_incertain'].astype('int8')

# Pour les effectifs, on reste sur du int64 ou int32 selon la taille des entreprises
df_clean['effectifs'] = df_clean['effectifs'].astype('int32')

# 5. La date reste inchangée (déjà bien typée)
# df_clean['Date_fermeture_finale'] = pd.to_datetime(df_clean['Date_fermeture_finale'])

print("✅ Typage terminé avec succès.")
print(f"📊 Nouvelle occupation mémoire : {df_clean.memory_usage().sum() / 1024**2:.2f} MB")
print("-" * 30)
print(df_clean.dtypes)

---

### 11. Noms des colonnes

In [ ]:
# --- RENOMMAGE DES COLONNES ---

mapping_colonnes = {
    'siren': 'SIREN',
    'denominationunitelegale': 'Dénomination',
    'codepostaletablissement': "Code postal de l'établissement",
    'codecommuneetablissement': "Code commune de l'établissement",
    'categoriejuridiqueunitelegale': "Catégorie juridique de l'unité légale",
    'activiteprincipaleunitelegale': "Activité principale de l'unité légale",
    'economiesocialesolidaireunitelegale': "Economie sociale et solidaire unité légale",
    'codedepartementetablissement': "Code du département de l'établissement",
    'coderegionetablissement': "Code de la région de l'établissement",
    'effectifs': 'Tranche_effectif_num',
    'est_fermee': 'fermeture'
}

# Application du renommage
df_clean = df_clean.rename(columns=mapping_colonnes)

print("✅ Colonnes renommées.")
print("-" * 30)

# Affichage des dimensions
print(f"📊 Dimensions du dataset : {df_clean.shape[0]:,} lignes x {df_clean.shape[1]} colonnes")
print("-" * 30)

# Affichage des types de données
print("🧬 Types des colonnes :")
print(df_clean.dtypes)
print("-" * 30)

# Aperçu
display(df_clean.head())

---

In [ ]:
# --- CONTRÔLE COMPLET DU DATASET (6M de lignes) ---

# On force l'affichage de toutes les lignes du résumé
pd.set_option('display.max_rows', None)

def bilan_complet_dataset(df):
    # 1. Calcul des statistiques de base
    bilan = pd.DataFrame({
        'Type': df.dtypes,
        'Non-Nuls': df.count(),
        'NaN': df.isna().sum(),
        '% NaN': (df.isna().sum() / len(df) * 100).round(2),
        'Valeurs Uniques': df.nunique()
    })
    
    print(f"📊 BILAN STRUCTUREL DU DATASET ({len(df):,} lignes)")
    print("-" * 50)
    print(bilan)
    print("-" * 50)
    
    # 2. Vérification de la cohérence Date vs Fermeture
    # Pour voir si le nombre de NaT en date correspond bien aux boîtes ouvertes (fermeture=0)
    ouvertes = df[df['fermeture'] == 0].shape[0]
    date_nat = df['Date_fermeture_finale'].isna().sum()
    
    print(f"💡 Cohérence Date/Statut :")
    print(f"   - Entreprises actives (fermeture=0) : {ouvertes:,}")
    print(f"   - Dates de fermeture absentes (NaT) : {date_nat:,}")
    
    if ouvertes == date_nat:
        print("   ✅ Parfait : Chaque entreprise active n'a logiquement pas de date de fermeture.")
    else:
        print("   ⚠️ Écart détecté : Certaines entreprises actives ont une date, ou inversement.")

# Exécution
bilan_complet_dataset(df_clean)

### 12. Export Parquet

In [ ]:
# import os

# # --- EXPORTATION FINALE ---
# file_path = "dataset_entreprises_6M_clean.parquet"

# # Utilisation de pyarrow pour une performance maximale sur 6M de lignes
# try:
#     df_clean.to_parquet(
#         file_path, 
#         engine='pyarrow', 
#         compression='snappy', 
#         index=False
#     )
    
#     # Vérification du résultat
#     file_size = os.path.getsize(file_path) / (1024 * 1024)
#     print(f"✅ Exportation terminée avec succès !")
#     print(f"📂 Fichier : {file_path}")
#     print(f"💾 Taille finale sur disque : {file_size:.2f} MB")
#     print(f"📊 Lignes exportées : {len(df_clean):,}")

# except Exception as e:
#     print(f"❌ Erreur lors de l'exportation : {e}")